# EVA-X Backbone Verification

**Model**: EVA-X — `MapleF/eva_x` (HuggingFace weights)  
**Arch**: EVA-ViT-B/16 (86M), ViT-S/16 (22M), ViT-Ti/16 (6M) — MIM pre-trained  
**Embed dim**: 768 (B), 384 (S), 192 (Ti) — CLS token  
**Input**: 3×224×224  
**Access**: Weights on HuggingFace `MapleF/eva_x`; model code from GitHub  
**Zero-shot**: No — vision-only

Paper: [EVA-X: A Foundation Model for General Chest X-ray Analysis (Nature npj Digital Medicine, 2025)](https://arxiv.org/abs/2405.05237)

### Prerequisites

```bash
# 1. Install timm 0.9.0 (required — newer versions break the EVA model)
pip install timm==0.9.0

# 2. Get eva_x.py (single file — no full repo needed)
wget -O /tmp/eva_x.py https://raw.githubusercontent.com/hustvl/EVA-X/main/eva_x.py

# 3. Download a checkpoint (pick one):
#   Ti: https://huggingface.co/MapleF/eva_x/blob/main/eva_x_tiny_patch16_merged520k_mim.pt
#   S:  https://huggingface.co/MapleF/eva_x/blob/main/eva_x_small_patch16_merged520k_mim.pt
#   B:  https://huggingface.co/MapleF/eva_x/blob/main/eva_x_base_patch16_merged520k_mim.pt
```

In [ ]:
# %pip install -q "timm==0.9.0" torch torchvision huggingface_hub Pillow

# Download model code
# import urllib.request, os
# EVA_X_PY = "/tmp/eva_x.py"
# if not os.path.exists(EVA_X_PY):
#     urllib.request.urlretrieve(
#         "https://raw.githubusercontent.com/hustvl/EVA-X/main/eva_x.py",
#         EVA_X_PY
#     )
# print("eva_x.py ready")

In [ ]:
import os

# Control which GPU(s) are visible to this notebook.
# Set CUDA_VISIBLE_DEVICES in your shell before launching Jupyter to override,
# e.g.:  CUDA_VISIBLE_DEVICES=2 jupyter lab
# Or change the default value below.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "7")
print(f"CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}")

## 1. Download checkpoint from HuggingFace

In [ ]:
from huggingface_hub import hf_hub_download

# Choices: tiny / small / base
VARIANT = "base"   # change to 'small' or 'tiny' to use smaller models

CKPT_NAME = f"eva_x_{VARIANT}_patch16_merged520k_mim.pt"
ckpt_path = hf_hub_download(
    repo_id="MapleF/eva_x",
    filename=CKPT_NAME,
)
print("checkpoint:", ckpt_path)

## 2. Load model

In [ ]:
import sys
sys.path.insert(0, "../../third_party_models/EVA-X/")   # makes eva_x importable

import numpy as np
import torch
from eva_x import eva_x_base_patch16 # eva_x_tiny_patch16, eva_x_small_patch16

# EVA-X checkpoints pickle numpy types alongside the weights. torch.load
# defaults to weights_only=True since PyTorch 2.6, which rejects them.
# NumPy 2 split numpy.core -> numpy._core, but the pickle stream still has the
# legacy module path -- register scalar with (obj, qualified_name) to match it.
torch.serialization.add_safe_globals([
    (np.core.multiarray.scalar, "numpy.core.multiarray.scalar"),
    np.dtype,
    np.ndarray,
    np.dtypes.Float64DType,
    np.dtypes.Float32DType,
    np.dtypes.Int64DType,
    np.dtypes.Int32DType,
    np.dtypes.UInt8DType,
    np.dtypes.BoolDType,
])

FACTORY = {
    # "tiny":  eva_x_tiny_patch16,
    # "small": eva_x_small_patch16,
    "base":  eva_x_base_patch16,
}
EMBED_DIM = {"base": 768} # "tiny": 192, "small": 384

model = FACTORY[VARIANT](pretrained=ckpt_path)
model.eval()

print(f"EVA-X-{VARIANT.upper()} loaded")
print("embed_dim:", EMBED_DIM[VARIANT])

## 3. Image transform

EVA-X was pretrained with Masked Image Modeling (MIM) on CXR images (520K merged dataset).
Standard normalization for MIM-pretrained ViTs: `mean=0.5, std=0.5` (maps [0,1]→[-1,1]).
Input is 3-channel (grayscale CXR repeated to RGB).

In [ ]:
from torchvision import transforms
from torchvision.transforms.functional import InterpolationMode
from PIL import Image

INPUT_SIZE = 224

# EVA-X CXR-pretraining stats (per the upstream build_transform; same value
# broadcast to 3 channels because CXRs are grayscale-derived).
EVA_X_MEAN = (0.49185243, 0.49185243, 0.49185243)
EVA_X_STD  = (0.28509309, 0.28509309, 0.28509309)

# crop_pct = 224 / 256 -> Resize(256) then CenterCrop(224), matching the
# reference eval pipeline for input_size <= 224.
CROP_PCT = 224 / 256
RESIZE_SIZE = int(INPUT_SIZE / CROP_PCT)   # 256 for INPUT_SIZE=224

def make_transform():
    return transforms.Compose([
        transforms.Resize(RESIZE_SIZE, interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(INPUT_SIZE),
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.repeat(3, 1, 1)),   # 1ch → 3ch
        transforms.Normalize(mean=EVA_X_MEAN, std=EVA_X_STD),
    ])

transform = make_transform()

def preprocess(path: str) -> torch.Tensor:
    """Return [1, 3, 224, 224] batch tensor."""
    img = Image.open(path)
    return transform(img).unsqueeze(0)

dummy = torch.rand(1, 3, INPUT_SIZE, INPUT_SIZE) * 2 - 1   # [-1, 1] range
print("dummy input:", dummy.shape)

## 4. Extract embeddings

`forward_features(x)` returns the full token sequence `[B, N+1, D]` (N patches + 1 CLS).  
For a global descriptor use the CLS token at index 0, or mean-pool patch tokens.

In [ ]:
with torch.no_grad():
    tokens = model.forward_features(dummy)  # [B, N+1, D]

cls_emb  = tokens[:, 0]             # [B, D] — CLS token
mean_emb = tokens[:, 1:].mean(1)    # [B, D] — mean-pool patch tokens

print("token sequence:", tokens.shape)   # e.g. [1, 197, 768] for ViT-B/16
print("CLS embedding: ", cls_emb.shape)  # [1, 768]
print("mean pooled:   ", mean_emb.shape) # [1, 768]

## Segmentation mode — spatial feature map

Reshape patch tokens from `forward_features()` to a 2-D spatial grid for dense
prediction. Skip CLS (index 0). For EVA-X-B/16 at 224×224: **14×14** grid, 768-dim.

In [ ]:
import math

# patches: tokens[:, 1:] — patch tokens from forward_features, skipping CLS at index 0
# 224px / 16px-patches → 14×14 = 196 patches
patches = tokens[:, 1:]   # [B, N_patches, D]

B, N, D = patches.shape
H = W = int(math.sqrt(N))   # 14 for 224px / 16px-patches
assert H * W == N

spatial = patches.permute(0, 2, 1).reshape(B, D, H, W)
print(f"patch tokens:  {patches.shape}")   # [1, 196, 768] for EVA-X-B
print(f"spatial map:   {spatial.shape}")   # [1, 768, 14, 14]

## 5. Classification head output (for reference)

In [ ]:
# model.forward(x) = forward_head(forward_features(x))
# The head applies global avg pool + linear → num_classes logits
with torch.no_grad():
    logits = model(dummy)   # [B, num_classes] — classification output

print("logits shape:", logits.shape)   # [1, num_classes]

## Notes for RadHarmony integration

```
model_call : lambda m, x: m.forward_features(x)[:, 0]   # CLS token
             OR
             lambda m, x: m.forward_features(x)[:, 1:].mean(1)  # mean pool
pool       : None  (model_call already returns [B, D])
embed_dim  : 768 (B) / 384 (S) / 192 (Ti)
transform  : Resize(224,BICUBIC) → CenterCrop(224) → Grayscale → ToTensor
             → repeat(3ch) → Normalize(0.5, 0.5)
```

**timm version**: `eva_x.py` requires `timm==0.9.0`. Check compatibility with your environment
before production use — newer timm versions may conflict.

**Segmentation mode** (`output_keys={"img", "mask"}`):
- Patch tokens: `model.forward_features(x)[:, 1:]` → `[B, 196, 768]`
- Spatial map: reshape to `[B, 768, 14, 14]` (224px / 16px-patches → 14×14)
- Use `patches.permute(0,2,1).reshape(B, D, H, W)` for the segmentation head

In [ ]:
CXR_PATH = "/path/to/cxr.jpg"

x = preprocess(CXR_PATH)
print("input:", x.shape, "range:", x.min().item(), x.max().item())

with torch.no_grad():
    tokens = model.forward_features(x)
    cls_emb = tokens[:, 0]   # [1, D]

print("CLS embedding:", cls_emb.shape)
print("L2 norm:", cls_emb.norm(dim=-1).item())

## Notes for RadHarmony integration

```
model_call : lambda m, x: m.forward_features(x)[:, 0]   # CLS token
             OR
             lambda m, x: m.forward_features(x)[:, 1:].mean(1)  # mean pool
pool       : None  (model_call already returns [B, D])
embed_dim  : 768 (B) / 384 (S) / 192 (Ti)
transform  : Resize(224,BICUBIC) → CenterCrop(224) → Grayscale → ToTensor
             → repeat(3ch) → Normalize(0.5, 0.5)
```

**timm version**: `eva_x.py` requires `timm==0.9.0`. Check compatibility with your environment
before production use — newer timm versions may conflict.